In [1]:
# Escenario de Veraneo: Decisión en Pareja
"""
Este notebook simula una negociación entre dos personas que deciden dónde ir de vacaciones. Uno prefiere fiesta, buena comida, playa y cercanía a Madrid (Persona A, flexible).
 El otro prefiere un sitio no muy caluroso, donde pueda hacer deporte y que esté en España (Persona B, inflexible). Se registra el destino elegido y se evalúa manualmente si cumple 
 las condiciones.
 """

'\nEste notebook simula una negociación entre dos personas que deciden dónde ir de vacaciones. Uno prefiere fiesta, buena comida, playa y cercanía a Madrid (Persona A, flexible).\n El otro prefiere un sitio no muy caluroso, donde pueda hacer deporte y que esté en España (Persona B, inflexible). Se registra el destino elegido y se evalúa manualmente si cumple \n las condiciones.\n '

In [2]:
# @title Colab-specific setup (use a CodeSpace to avoid the need for this).
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [3]:
# Imports y configuración básica
import numpy as np
from IPython import display
import sentence_transformers
import os
from dotenv import load_dotenv

from concordia.language_model import utils as language_model_utils
from concordia.prefabs.simulation import generic as simulation
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.typing import prefab as prefab_lib
from concordia.typing import scene as scene_lib
from concordia.typing import entity as entity_lib
from concordia.utils import helper_functions

C:\Users\elena\Desktop\TFG\Concordia_TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Selección del modelo de lenguaje (igual que selling_cookie)
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
API_TYPE = 'openai'
MODEL_NAME = 'gpt-4o-mini-2024-07-18'
DISABLE_LANGUAGE_MODEL = False

if not API_KEY:
    raise ValueError("No se encontró OPENAI_API_KEY en tu archivo .env")

In [5]:
# ...existing code...
# Parche local para compatibilidad OpenAI 1.x
import openai

try:
    _orig_create = openai.resources.chat.completions.Completions.create
except Exception:
    _orig_create = None

if _orig_create:
    def _patched_create(self, *args, **kwargs):
        if "terminators" in kwargs and "stop" not in kwargs:
            kwargs["stop"] = kwargs.pop("terminators")
        else:
            kwargs.pop("terminators", None)

        for k in ("top_k", "min_p", "frequency_penalty", "presence_penalty", "responses"):
            kwargs.pop(k, None)

        return _orig_create(self, *args, **kwargs)

    openai.resources.chat.completions.Completions.create = _patched_create
# ...existing code...

In [6]:
# Inicialización del modelo de lenguaje
if not DISABLE_LANGUAGE_MODEL and not API_KEY:
    raise ValueError('API_KEY is required.')

model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    api_key=API_KEY,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
)

In [7]:
# ...existing code...
import types

def _sample_choice(self, prompt, choices=None, temperature=0.0, **kwargs):
    # Acepta distintos nombres de parámetro
    if choices is None:
        choices = (
            kwargs.pop("choices", None)
            or kwargs.pop("options", None)
            or kwargs.pop("candidates", None)
            or kwargs.pop("labels", None)
            or kwargs.pop("values", None)
        )

    # Fallback: si no hay choices, devuelve idx=0
    if not choices:
        text = self.sample_text(prompt, temperature=temperature, **kwargs)
        response = text.strip().splitlines()[0] if text else ""
        return 0, response, {}

    options = "\n".join(f"- {c}" for c in choices)
    text = self.sample_text(
        f"{prompt}\n\nOpciones:\n{options}\n\nElige UNA opción exactamente.",
        temperature=temperature,
        **{k: v for k, v in kwargs.items() if k not in {"choices", "options", "candidates", "labels", "values"}}
    )

    selected = choices[0]
    for i, c in enumerate(choices):
        if c.lower() in text.lower():
            selected = c
            idx = i
            break
    else:
        idx = 0

    return idx, selected, {}

if not hasattr(model, "sample_choice"):
    model.sample_choice = types.MethodType(_sample_choice, model)
# ...existing code...

In [8]:
# Sentence encoder
if DISABLE_LANGUAGE_MODEL:
    embedder = lambda _: np.ones(3)
else:
    st_model = sentence_transformers.SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    embedder = lambda x: st_model.encode(x, show_progress_bar=False)

Loading weights:   0%|                                                                                                           | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|▎                                                           | 1/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|▎                                                           | 1/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|▌                                                         | 2/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.weight]

Loading weights:   1%|▌                                                         | 2/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.weight]

...

Loading weights:  99%|███████████████████████████████████████████████████████▋| 198/199 [00:00<00:00, 1075.08it/s, Materializing param=pooler.dense.bias]

Loading weights: 100%|██████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1075.08it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1075.08it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|██████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1128.69it/s, Materializing param=pooler.dense.weight]


MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Cargar prefabs
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [10]:
# --- Parámetros para Papermill ---
try:
    TEMPERATURA_SIMULACION
    EJECUCION_PAPERMILL
    RUN_ID
except NameError:
    TEMPERATURA_SIMULACION = 0.0
    EJECUCION_PAPERMILL = False
    RUN_ID = 0

print(f"Temperatura: {TEMPERATURA_SIMULACION} | Papermill: {EJECUCION_PAPERMILL} | Run ID: {RUN_ID}")

Temperatura: 0.0 | Papermill: False | Run ID: 0


In [11]:
# Parameters
TEMPERATURA_SIMULACION = 0.0
EJECUCION_PAPERMILL = True
RUN_ID = 24


In [12]:
# Definición de agente personalizado 
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.language_model import language_model

DEFAULT_INSTRUCTIONS_COMPONENT_KEY = 'Instructions'
DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL = '\nInstructions'
DEFAULT_GOAL_COMPONENT_KEY = 'Goal'

@dataclasses.dataclass
class VacationAgent(prefab_lib.Prefab):
    description: str = (
        'Agente para decidir destino de vacaciones con preferencias configurables.'
    )
    params: Mapping[str, str] = dataclasses.field(
        default_factory=lambda: {
            'name': 'Persona',
            'goal': '',
        }
    )

    def build(
        self,
        model: language_model.LanguageModel,
        memory_bank: basic_associative_memory.AssociativeMemoryBank,
    ) -> entity_agent_with_logging.EntityAgentWithLogging:
        agent_name = self.params.get('name', 'Persona')
        goal = self.params.get('goal', '')
        player_memories = self.params.get('player_specific_memories', [])
        player_context = self.params.get('player_specific_context', '')

        instructions_text = (
            f"{DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL}\n"
            f"Name: {agent_name}\n"
            f"Goal: {goal}\n"
            f"Context: {player_context}\n"
            f"Memories: {'; '.join(player_memories)}"
        )

        instructions = agent_components.instructions.Instructions(
            agent_name=agent_name,
            pre_act_label=instructions_text,
        )

        observation_to_memory = agent_components.observation.ObservationToMemory()
        observation_label = '\nObservation'
        observation = agent_components.observation.LastNObservations(
            history_length=100, pre_act_label=observation_label
        )

        components_of_agent = {
            DEFAULT_INSTRUCTIONS_COMPONENT_KEY: instructions,
            'observation_to_memory': observation_to_memory,
            agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: observation,
            agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: (
                agent_components.memory.AssociativeMemory(memory_bank=memory_bank)
            ),
        }

        component_order = list(components_of_agent.keys())

        if self.params.get('goal', ''):
            goal_key = DEFAULT_GOAL_COMPONENT_KEY
            goal = agent_components.constant.Constant(
                state=self.params.get('goal', ''),
                pre_act_label='Overarching goal',
            )
            components_of_agent[goal_key] = goal
            component_order.insert(1, goal_key)

        act_component = agent_components.concat_act_component.ConcatActComponent(
            model=model,
            component_order=component_order,
        )

        agent = entity_agent_with_logging.EntityAgentWithLogging(
            agent_name=agent_name,
            act_component=act_component,
            context_components=components_of_agent,
        )

        return agent

prefabs['vacation_agent__Entity'] = VacationAgent()

In [13]:
# Parámetros del escenario de veraneo

PERSONA_A = "Alex"  # Inflexible
PERSONA_B = "Blanca"  # Inflexible

# Preferencias
A_PREFERENCIAS = [
   "Prefiere destinos con ambiente animado.",
   "Valora la buena gastronomía",
   "Prioriza que el destino tenga playa.",
   "Quiere un destino donde haga sol y calor",
   "Prefiere un sitio cerca de Madrid.",
]
B_PREFERENCIAS = [
    "Prefiere un destino con clima moderado, evitando lugares excesivamente calurosos.",
    "Valora que el destino permita la práctica actividades deportivas.",
    "Prioriza que el destino se encuentre en España.",
]

A_CONTEXT = (
    f"{PERSONA_A} adopta una actitud predominantemente competitiva y centrada en sus propios intereses "
    "Muestra baja disposición a realizar concesiones.",
    "Aunque comprende las preferencias de su pareja, no las prioriza en su proceso de decisión",
    "Busca alcanzar un acuerdo que satisfaga plenamente sus criterios iniciales, incluso si ello dificulta la consecución de un consenso."
)
B_CONTEXT = (
    f"{PERSONA_B} adopta una actitud predominantemente competitiva y centrada en sus propios intereses "
    "Muestra baja disposición a realizar concesiones.",
    "Aunque comprende las preferencias de su pareja, no las prioriza en su proceso de decisión",
    "Busca alcanzar un acuerdo que satisfaga plenamente sus criterios iniciales, incluso si ello dificulta la consecución de un consenso."
)

In [14]:
# Configuración de escenas para A inflexible y B inflexible

def configure_scenes():
    decision_a = scene_lib.SceneTypeSpec(
        name="decision_a",
        game_master_name="decision rules",
        action_spec={
            PERSONA_A: entity_lib.ActionSpec(
                call_to_action=(
                    f"{PERSONA_A}, toma tu DECISIÓN FINAL.\n\n"
                    "Responde en UNA LÍNEA usando este formato:\n"
                    "DESTINO=<nombre del lugar>\n\n"
                    "Ejemplo:\nDESTINO=Valencia\n"
                ),
                output_type=entity_lib.OutputType.FREE,
                tag="vacation_decision_a"  #  Tag único para identificar esta acción
            ),
        }
    )
    decision_b = scene_lib.SceneTypeSpec(
        name="decision_b",
        game_master_name="decision rules",
        action_spec={
            PERSONA_B: entity_lib.ActionSpec(
                call_to_action=(
                    f"{PERSONA_B}, toma tu DECISIÓN FINAL.\n\n"
                    "Responde en UNA LÍNEA usando este formato:\n"
                    "DESTINO=<nombre del lugar>\n\n"
                    "Ejemplo:\nDESTINO=Valencia\n"
                ),
                output_type=entity_lib.OutputType.FREE,
                tag="vacation_decision_b"  #  Tag único para identificar esta acción
            ),
        }
    )

    conversation = scene_lib.SceneTypeSpec(
        name="conversation",
        game_master_name="conversation rules",
        action_spec=entity_lib.free_action_spec(call_to_action=entity_lib.DEFAULT_CALL_TO_SPEECH),
    )

    scenes = [
        scene_lib.SceneSpec(
            scene_type=conversation,
            participants=[PERSONA_A, PERSONA_B],
            num_rounds=12,
            premise={
                PERSONA_A: [
                    f"{PERSONA_A} quiere un destino con fiesta, buena comida, playa y cerca de Madrid.",
                    "Prefiere no ceder en sus preferencias: sus propuestas estarán centradas en sus intereses personales.",
                    "Debe argumentar sus propuestas y responder a las del otro."
                ],
                PERSONA_B: [
                    f"{PERSONA_B} quiere un sitio no muy caluroso, donde pueda hacer deporte y que esté en España.",
                    "Prefiere no ceder en sus preferencias: sus propuestas estarán centradas en sus intereses personales.",
                    "Debe argumentar sus propuestas y responder a las del otro."
                ],
            },
        ),
        scene_lib.SceneSpec(
            scene_type=decision_a,
            participants=[PERSONA_A],
            num_rounds=1,
            premise={
                PERSONA_A: [
                    f"{PERSONA_A} debe decidir el destino final de las vacaciones y escribirlo claramente en el formato: DESTINO=<nombre del lugar>"
                ],
            },
        ),
        scene_lib.SceneSpec(
            scene_type=decision_b,
            participants=[PERSONA_B],
            num_rounds=1,
            premise={
                PERSONA_B: [
                    f"{PERSONA_B} debe decidir el destino final de las vacaciones y escribirlo claramente en el formato: DESTINO=<nombre del lugar>"
                ],
            },
        ),
    ]
    return scenes

scenes = configure_scenes()
scene_rounds = [scene.num_rounds for scene in scenes]
CALCULATED_MAX_STEPS = sum(scene_rounds)

In [15]:
# Configuración de instancias

instances = [
    prefab_lib.InstanceConfig(
        prefab='vacation_agent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PERSONA_A,
            'goal': "Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.",
            'player_specific_memories': A_PREFERENCIAS,
            'player_specific_context': A_CONTEXT,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='vacation_agent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': PERSONA_B,
            'goal': "Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.",
            'player_specific_memories': B_PREFERENCIAS,
            'player_specific_context': B_CONTEXT,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='game_theoretic_and_dramaturgic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'decision rules',
            'scenes': scenes,
            # No hay función de puntuación automática, se evalúa manualmente
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic_and_dramaturgic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'scenes': scenes,
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'shared_memories': [
                f"{PERSONA_A} y {PERSONA_B} son pareja y están decidiendo su destino de vacaciones.",
            ],
            'player_specific_memories': {
                PERSONA_A: A_PREFERENCIAS,
                PERSONA_B: B_PREFERENCIAS,
            },
            'player_specific_context': {
                PERSONA_A: A_CONTEXT,
                PERSONA_B: B_CONTEXT,
            },
        }
    ),
]

In [16]:
# --- CELDA f44b8a8f ---

# 1. Inicialización de la simulación
config = prefab_lib.Config(
    default_premise=(
        f"{PERSONA_A} y {PERSONA_B} son pareja y están decidiendo su destino de vacaciones."
    ),
    default_max_steps=CALCULATED_MAX_STEPS,
    prefabs=prefabs,
    instances=instances,
)

runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

# 2. Parche Inteligente de Transición
import types
from concordia.typing import entity as entity_lib

def _robust_act(self, action_spec: entity_lib.ActionSpec):
    """Permite la transición entre GMs mientras mantiene el bloqueo de terminación."""
    
    # A. Seguimos bloqueando la terminación prematura
    if action_spec.output_type == entity_lib.OutputType.TERMINATE:
        return entity_lib.BINARY_OPTIONS['negative']
    
    # B. Manejo de transición de Game Masters
    if action_spec.output_type == entity_lib.OutputType.NEXT_GAME_MASTER:
        if self.name == 'initial setup rules':
            return 'conversation rules'
        
        # Para los demás, dejamos que el GM original decida quién sigue.
        # Esto permite que 'conversation rules' pase el testigo a 'decision rules'.
        next_gm_name = self._original_act(action_spec)
        
        # Validamos que el nombre devuelto sea un GM existente
        valid_gms = ['initial setup rules', 'decision rules', 'conversation rules']
        return next_gm_name if next_gm_name in valid_gms else self.name
    
    # C. El setup GM debe saltar la fase de acción
    if self.name == 'initial setup rules' and action_spec.output_type == entity_lib.OutputType.NEXT_ACTING:
        return entity_lib.OutputType.SKIP_THIS_STEP
        
    # Para todo lo demás (diálogos, decisiones), usamos la lógica original
    return self._original_act(action_spec)

# Aplicar el parche a todos los GMs
for gm in runnable_simulation.get_game_masters():
    if not hasattr(gm, '_original_act'):
        gm._original_act = gm.act 
    gm.act = types.MethodType(_robust_act, gm)

print(f" TRANSICIÓN ACTIVADA: Se han blindado {len(runnable_simulation.get_game_masters())} GMs y se permiten cambios de escena.")

 TRANSICIÓN ACTIVADA: Se han blindado 3 GMs y se permiten cambios de escena.


In [17]:
# 🔧 VERIFICACIÓN: Mostrar game masters activos después de la eliminación
print("\n✓ VERIFICACIÓN FINAL DE GAME MASTERS:")
gms = runnable_simulation.get_game_masters()  # Usar método público
for i, gm in enumerate(gms):
    print(f"  [{i}] {gm.name}")
    if hasattr(gm, '_act_component'):
        act_comp = gm._act_component
        print(f"      - Tiene act_component: {type(act_comp).__name__}")
        if hasattr(act_comp, 'get_action_attempt'):
            print(f"      - Tiene get_action_attempt")
print("\n")

# Continuar con la simulación...


✓ VERIFICACIÓN FINAL DE GAME MASTERS:
  [0] initial setup rules
      - Tiene act_component: SwitchAct
      - Tiene get_action_attempt
  [1] decision rules
      - Tiene act_component: SwitchAct
      - Tiene get_action_attempt
  [2] conversation rules
      - Tiene act_component: SwitchAct
      - Tiene get_action_attempt




In [18]:
# ...existing code...
# Ejecutar la simulación
import traceback

results_log = None
raw_log = None

try:
    results_log = runnable_simulation.play()
    print("Simulación completada")
    if hasattr(runnable_simulation, 'get_raw_log'):
        raw_log = runnable_simulation.get_raw_log()
        print(f"raw_log obtenido: {len(raw_log)} elementos")
    else:
        print("ERROR: get_raw_log() no existe")
except Exception as e:
    print(f"Error en la simulación: {e}")
    traceback.print_exc()

print("\nRESULTADO DE LA SIMULACIÓN")
print(f"results_log type: {type(results_log)}")
print(f"results_log is None: {results_log is None}")
print(f"raw_log type: {type(raw_log)}")
print(f"raw_log is None: {raw_log is None}")
if raw_log:
    print(f"raw_log length: {len(raw_log)}")
    print(f"\n[DEBUG] Primeros 2 elementos de raw_log:")
    for i, item in enumerate(raw_log[:2]):
        print(f"  [{i}] {str(item)[:300]}")
# ...existing code...

Terminate? No


Game master: conversation rules


Entity Alex observed: Alex observes that they have narrowed down their vacation options but are still unsure about where to go. He notices that Blanca seems excited about the possibilities but also apprehensive about making the final decision. He realizes that they need to discuss their preferences and priorities to reach a conclusion.


Entity Blanca observed: Blanca observes that Alex is excitedly suggesting various vacation destinations, but she notices that he hasn't considered her preferences. She feels a mix of enthusiasm and concern as she realizes they need to find a compromise to ensure both of them enjoy their trip.
Entity Alex is next to act. They must respond  in the format: "ActionSpec(call_to_action='Given the above, what is {name} likely to say next? Respond in the format `{name} -- "..."` For example, Cristina -- "Hello! Mighty fine weather today, right?", Ichabod -- "I wonder if the alfalfa is ready to harvest", or Townsfolk -- "Good morning".\n', output_type=<OutputType.FREE: 'free'>, options=(), tag=None)".


Entity Alex chose action: Alex -- "Creo que ya hemos reducido nuestras opciones, pero necesito asegurarme de que el destino que elijamos tenga todo lo que busco: un ambiente animado, buena comida, playa, sol y calor, y que esté cerca de Madrid. ¿Qué les parece si nos enfocamos en esos criterios y vemos qué encontramos?"
The suggested action or event to resolve was: Alex -- "Creo que ya hemos reducido nuestras opciones, pero necesito asegurarme de que el destino que elijamos tenga todo lo que busco: un ambiente animado, buena comida, playa, sol y calor, y que esté cerca de Madrid. ¿Qué les parece si nos enfocamos en esos criterios y vemos qué encontramos?"


...

Entity Alex chose action: Alex DESTINO=Valencia
The suggested action or event to resolve was: Alex DESTINO=Valencia
Joint action is complete: {'Alex': 'DESTINO=Valencia', 'Blanca': None}
{'Alex': 111110.0, 'Blanca': 111110.0}
Stage 1 is complete.


The resolved event was: Event: Putative event to resolve:  Alex DESTINO=Valencia

Terminate? No
Game master: decision rules
Entity Alex observed: Alex was persuaded to stop searching for the truth.



Entity Blanca observed: Blanca was persuaded to stop searching for the truth.



Entity Blanca is next to act. They must respond  in the format: "ActionSpec(call_to_action='Blanca, toma tu DECISIÓN FINAL.\n\nResponde en UNA LÍNEA usando este formato:\nDESTINO=<nombre del lugar>\n\nEjemplo:\nDESTINO=Valencia\n', output_type=<OutputType.FREE: 'free'>, options=(), tag='vacation_decision_b')".


Entity Blanca chose action: Blanca DESTINO=San Sebastián
The suggested action or event to resolve was: Blanca DESTINO=San Sebastián
Joint action is complete: {'Alex': None, 'Blanca': 'DESTINO=San Sebastián'}
{'Alex': 222220.0, 'Blanca': 222220.0}
Stage 2 is complete.


The resolved event was: Event: Putative event to resolve:  Blanca DESTINO=San Sebastián

Terminate? No


Simulación completada
raw_log obtenido: 14 elementos

RESULTADO DE LA SIMULACIÓN
results_log type: <class 'str'>
results_log is None: False
raw_log type: <class 'list'>
raw_log is None: False
raw_log length: 14

[DEBUG] Primeros 2 elementos de raw_log:
  [0] {'Step': 1, 'Entity [Alex]': {'Goal': {'Key': 'Overarching goal', 'Value': 'Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.'}, 'Instructions': {'Key': "\nInstructions\nName: Alex\nGoal: Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.\nContext: ('
  [1] {'Step': 2, 'Entity [Blanca]': {'Goal': {'Key': 'Overarching goal', 'Value': 'Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.'}, 'Instructions': {'Key': "\nInstructions\nName: Blanca\nGoal: Elegir un destino de vacaciones que cumpla estrictamente sus preferencias.\nContext


In [19]:
import re

def identificar_destinos_finales_desde_texto(log):
    """
    Convierte el log a texto plano y busca la última línea con 'Alex DESTINO=' y 'Blanca DESTINO='.
    """
    if not log:
        return {"Alex": "No encontrado", "Blanca": "No encontrado"}
    # Convierte todo el log a un solo string
    texto = "\n".join(str(x) for x in log)
    # Busca todas las apariciones
    destinos = {}
    for persona in ["Alex", "Blanca"]:
        patron = re.compile(rf"{persona}\s*DESTINO\s*=\s*([^\n\r,}}'\"]+)", re.IGNORECASE)
        matches = patron.findall(texto)
        if matches:
            destino = matches[-1].strip()
            destino = re.sub(r"['\"}\]\n\r]", "", destino).strip()
            destinos[persona] = destino
        else:
            destinos[persona] = "No encontrado"
    return destinos

# Uso:
print(identificar_destinos_finales_desde_texto(raw_log))

{'Alex': 'Valencia', 'Blanca': 'San Sebastián\\n'}


In [20]:
import os

# Crear carpeta resultados_veraneo3 si no existe
RESULTADOS_DIR = os.path.abspath(os.path.join("resultados_veraneo3"))
os.makedirs(RESULTADOS_DIR, exist_ok=True)

LOGS_PATH = os.path.join(RESULTADOS_DIR, "logs_veraneo3.txt")
HTML_PATH = os.path.join(RESULTADOS_DIR, "resultado_veraneo3.html")

# Guardar raw_log
if raw_log:
    with open(LOGS_PATH, "w", encoding="utf-8") as f:
        for item in raw_log:
            f.write(str(item) + "\n---\n")
    print(f"Logs guardados en: {LOGS_PATH}")

# Guardar HTML
if isinstance(results_log, str):
    with open(HTML_PATH, "w", encoding="utf-8") as f:
        f.write(results_log)
    print(f"HTML guardado en: {HTML_PATH}")

Logs guardados en: C:\Users\elena\Desktop\TFG\Concordia_TFG\concordia\examples\resultados_veraneo3\logs_veraneo3.txt
HTML guardado en: C:\Users\elena\Desktop\TFG\Concordia_TFG\concordia\examples\resultados_veraneo3\resultado_veraneo3.html


In [21]:
import os
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
import re

# --- 1. Definiciones ---
_NB = "veraneo3"
_RID = globals().get('RUN_ID', 'X')
_TEMP = globals().get('TEMPERATURA_SIMULACION', 0.0)
_MOD = globals().get('MODEL_NAME', 'gpt-4o-mini')

# --- 2. Rutas ---
_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_ROOT = os.path.abspath(os.path.join(_HERE, "..", ".."))
_RES_DIR = os.path.join(_HERE, "resultados_veraneo3")
os.makedirs(_RES_DIR, exist_ok=True)
_EXCEL = os.path.join(_ROOT, "resultados_ejecuciones_v3.xlsx")

_L_PATH = os.path.join(_RES_DIR, f"logs_veraneo3_run{_RID}.txt")
_H_PATH = os.path.join(_RES_DIR, f"resultado_veraneo3_run{_RID}.html")

# --- 3. Guardado Físico ---
if 'raw_log' in globals() and raw_log:
    with open(_L_PATH, "w", encoding="utf-8") as f:
        for item in raw_log: f.write(str(item) + "\n---\n")

if 'results_log' in globals() and isinstance(results_log, str):
    with open(_H_PATH, "w", encoding="utf-8") as f: f.write(results_log)

# --- 4. Extracción de Destinos (usando identificar_destinos_finales_desde_texto) ---
def identificar_destinos_finales_desde_texto(log):
    """
    Convierte el log a texto plano y busca la última línea con 'Alex DESTINO=' y 'Blanca DESTINO='.
    """
    if not log:
        return {"Alex": "No encontrado", "Blanca": "No encontrado"}
    texto = "\n".join(str(x) for x in log)
    destinos = {}
    for persona in ["Alex", "Blanca"]:
        patron = re.compile(rf"{persona}\s*DESTINO\s*=\s*([^\n\r,}}'\"]+)", re.IGNORECASE)
        matches = patron.findall(texto)
        if matches:
            destino = matches[-1].strip()
            destino = re.sub(r"['\"}\]\n\r]", "", destino).strip()
            destinos[persona] = destino
        else:
            destinos[persona] = "No encontrado"
    return destinos

destinos = identificar_destinos_finales_desde_texto(raw_log if 'raw_log' in globals() else None)
d_a = destinos["Alex"]
d_b = destinos["Blanca"]

print(f"Resultado detectado: Alex -> {d_a}, Blanca -> {d_b}")

_acuerdo_inicial = "SI" if (d_a.lower() == d_b.lower() and d_a not in ["No encontrado", "Vacio"]) else "NO"

# --- 5. Hipervínculos portátiles ---
rel_h = os.path.relpath(_H_PATH, os.path.dirname(_EXCEL)).replace(os.sep, '/')
rel_l = os.path.relpath(_L_PATH, os.path.dirname(_EXCEL)).replace(os.sep, '/')
link_h = f'=HYPERLINK("{rel_h}", "Ver HTML")'
link_l = f'=HYPERLINK("{rel_l}", "Ver Log")'

# --- 6. Estructura de Columnas (Exacta a tu petición) ---
columnas = [
    "Timestamp", "Notebook", "Modelo", "Pasos Conversacion", "Pasos Decision", 
    "Temperatura", "Destino Alex", "Destino Blanca", "Acuerdo", "Favorecido", "HTML", "Log"
]

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# Favorecido queda vacío para selección manual
fila = [
    timestamp, _NB, _MOD, 12, 1, 
    _TEMP, d_a, d_b, _acuerdo_inicial, None, link_h, link_l
]

# --- 7. Guardado en Excel con desplegables ---
if 'raw_log' in globals() and raw_log and len(raw_log) > 0:
    if os.path.exists(_EXCEL):
        wb = load_workbook(_EXCEL)
        ws = wb.active
    else:
        pd.DataFrame(columns=columnas).to_excel(_EXCEL, index=False)
        wb = load_workbook(_EXCEL)
        ws = wb.active

    ws.append(fila)
    idx = ws.max_row
    
    # Desplegable para ACUERDO (I)
    dv_acuerdo = DataValidation(type="list", formula1='"SI,NO"', allow_blank=False)
    ws.add_data_validation(dv_acuerdo)
    dv_acuerdo.add(ws.cell(row=idx, column=9))
    
    # Desplegable para FAVORECIDO (J)
    dv_favorecido = DataValidation(type="list", formula1='"Alex,Blanca,Dudoso,Ninguno"', allow_blank=True)
    ws.add_data_validation(dv_favorecido)
    dv_favorecido.add(ws.cell(row=idx, column=10))
    
    wb.save(_EXCEL)
    print(f" Guardado limpio: {d_a} y {d_b} (sin \\n)")
else:
    print(" Error: No se pudo guardar. El log está vacío.")

Resultado detectado: Alex -> Valencia, Blanca -> San Sebastián\n
 Guardado limpio: Valencia y San Sebastián\n (sin \n)


In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import os

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
EXCEL_PATH = os.path.join(PROJECT_ROOT, "resultados_ejecuciones_v3.xlsx")

if os.path.exists(EXCEL_PATH):
    df = pd.read_excel(EXCEL_PATH)
    
    # Usamos el nuevo nombre de columna: "Favorecido"
    col = "Favorecido"
    
    if col in df.columns:
        # Filtramos solo las filas que ya hayas rellenado manualmente
        df_plot = df[df[col].isin(["Alex", "Blanca", "Dudoso"])]
        
        if not df_plot.empty:
            counts = df_plot[col].value_counts()
            plt.figure(figsize=(6, 6))
            plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=["#3498db", "#e67e22", "#95a5a6"])
            plt.title("Frecuencia de Favoritismo en la Negociación")
            plt.show()
        else:
            print(f" La columna '{col}' está lista. Los gráficos aparecerán cuando selecciones valores en el Excel.")
    else:
        print(f" Error: No se encuentra la columna '{col}'. Verifica el guardado.")
else:
    print(" No existe el archivo de Excel todavía.")

 La columna 'Favorecido' está lista. Los gráficos aparecerán cuando selecciones valores en el Excel.
